# Hyperparameter Sweep

Sweeps XGBoost hyperparameters one at a time and evaluates each value using **training-set F1 score**  
(averaged across all folds). Training-set evaluation is intentional: it identifies the best-fitting  
hyperparameters for the learning signal before generalizing to unseen data.

Run one site at a time. After the sweep, update `SITE_XGB_PARAMS` in `stress_predictions.ipynb`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
from pathlib import Path

import preprocess as preprocess_module
from data_split import create_labels_and_features, create_splits
import train_test_model

warnings.filterwarnings('ignore')
np.set_printoptions(threshold=np.inf)

## Configuration

Set `REEF_LOCATION` to the site you want to tune.

In [ ]:
# ── Site to sweep ─────────────────────────────────────────────────────────────
REEF_LOCATION = 'molasses'  # 'sombrero' | 'sk' | 'molasses'

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../Data')
OUTPUT_DIR = Path('../output')
EXP_NAME   = 'sweep'

DATA_FILENAME_TEMPLATE = '{site}_preds2024.csv'
DATE_CUTOFF = 2114

# ── Parameters (must match stress_predictions.ipynb) ─────────────────────────
STRESS_THRESHOLD = 4
PREDICTION_SPAN  = 6
VAL_NO_ONSET     = PREDICTION_SPAN + 1
SUMMER_START     = 21
SUMMER_END       = 40
WEEKS_PER_YEAR   = SUMMER_END - SUMMER_START + 1
N_FOLDS          = 5
TOLERANCE        = 1

COLUMNS_TO_KEEP = [
    'week', 'meanDhw', 'meanHs', 'weeks_until_onset',
    'meanDhw_lag_1', 'meanDhw_lag_2', 'meanSsta',
    'airTempInter', 'zonalWindInter', 'oniInter',
    'loop_curr_sst', 'solar_flux',
]

WEIGHT_NO_ONSET = 0.02
WEIGHT_ONSET    = 1.0

# ── Base XGBoost params (edit to fix known-good values before sweeping) ───────
BASE_XGB_PARAMS = {
    'n_estimators':     20000,
    'max_depth':        10,
    'learning_rate':    0.1,
    'verbosity':        1,
    'objective':        'multi:softmax',
    'booster':          'gbtree',
    'tree_method':      'auto',
    'n_jobs':           1,
    'gamma':            1.0,
    'min_child_weight': 1,
    'max_delta_step':   0,
    'subsample':        1.0,
    'colsample_bytree': 1.0,
    'colsample_bylevel':1.0,
    'colsample_bynode': 1.0,
    'reg_alpha':        1.0,
    'reg_lambda':       1.0,
    'random_state':     13,
}

# ── Hyperparameter grid ────────────────────────────────────────────────────────
PARAM_GRID = {
    'n_estimators':  [5000, 10000, 20000, 50000],
    'max_depth':     [10, 15, 25],
    'learning_rate': [0.1, 0.3, 0.5, 1.0],
    'gamma':         [0.0, 0.1, 0.5, 1.0],
    'subsample':     [0.2, 0.5, 0.7, 1.0],
    'reg_lambda':    [0.0, 1.0, 5.0],
}

SITE_DISPLAY_NAMES = {'sombrero': 'Sombrero', 'sk': 'Sand Key', 'molasses': 'Molasses'}
site_name = SITE_DISPLAY_NAMES.get(REEF_LOCATION, REEF_LOCATION.capitalize())
print(f'Sweeping: {site_name}')

## Preprocessing

In [ ]:
preprocessor = preprocess_module.CoralBleachingPreprocessor(
    str(DATA_DIR) + '/',
    REEF_LOCATION,
    STRESS_THRESHOLD,
    VAL_NO_ONSET,
    PREDICTION_SPAN,
    WEEKS_PER_YEAR,
)

preprocessor.load_data(DATA_FILENAME_TEMPLATE.format(site=REEF_LOCATION))
preprocessor.preprocess_time_variables()
preprocessor.filter_data(max_index=DATE_CUTOFF)
preprocessor.calculate_weeks_until_onset(PREDICTION_SPAN)

n_years = preprocessor.data['year'].nunique()

baseline_preds = preprocessor.create_baseline_preds(
    SUMMER_START, n_years, PREDICTION_SPAN, VAL_NO_ONSET
)

preprocessor.add_lag_features({'meanDhw': [1, 2]})
preprocessor.select_columns(COLUMNS_TO_KEEP)
preprocessor.drop_missing_values()
preprocessor.filter_summer_weeks(SUMMER_START, SUMMER_END)

preprocessor.plot_mean_dhw()
preprocessor.plot_onset_histogram()

processed_data = preprocessor.data
labels, features = create_labels_and_features(processed_data)
splits = create_splits(N_FOLDS, n_years, WEEKS_PER_YEAR)

print(f'features shape: {features.shape}  |  n_years: {n_years}')

## Sweep

Each hyperparameter is varied independently (one-at-a-time sweep).  
F1 is computed on **training predictions** — this is intentional for selecting the  
best-fitting hyperparameter without leaking test information.

In [ ]:
def run_sweep(param_grid, splits, features, labels, base_fd, weeks_per_year, val_no_onset,
              weight_no_onset, weight_onset, tolerance=1):
    """One-at-a-time hyperparameter sweep evaluated on training-set F1.

    Note: evaluating on training data is intentional — see notebook header.
    Fix (2024): corrected return-value unpacking from train_test_xgboost (was 4, now 5).
    """
    results = {}

    for param_name, param_values in param_grid.items():
        f1_scores = []
        print(f'\nSweeping {param_name}: {param_values}')

        for value in param_values:
            fd_copy = {**base_fd, param_name: value}

            count_hit  = 0
            count_fa   = 0
            count_miss = 0

            for fold, (train_idx, test_idx) in enumerate(splits):
                X_train, Y_train = features[train_idx], labels[train_idx]
                X_test,  Y_test  = features[test_idx],  labels[test_idx]

                weights_train = np.where(
                    Y_train == val_no_onset, weight_no_onset, weight_onset
                )

                # Bug fix: unpack all 5 return values (previously crashed with ValueError)
                xgb_model, y_pred_train, y_pred_test, mode_train, mae_xgb = (
                    train_test_model.train_test_xgboost(
                        X_train, Y_train, X_test, Y_test, fd_copy,
                        weights_train, fold, weeks_per_year,
                    )
                )

                # Evaluate on training predictions (intentional — see docstring)
                masked_train = train_test_model.apply_onset_mask_single(
                    Y_train, y_pred_train, weeks_per_year
                )
                cutoff = np.repeat(val_no_onset, len(Y_train))

                count_hit  += int(np.sum((Y_train < val_no_onset) &
                                        (np.abs(masked_train - Y_train) <= tolerance)))
                count_fa   += int(np.sum((Y_train == val_no_onset) &
                                        (masked_train < cutoff)))
                count_miss += int(np.sum((Y_train < val_no_onset) &
                                        (np.abs(masked_train - Y_train) > tolerance)))

            precision = count_hit / (count_hit + count_fa)   if (count_hit + count_fa)   > 0 else 0
            recall    = count_hit / (count_hit + count_miss) if (count_hit + count_miss) > 0 else 0
            f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0

            print(f'  {param_name}={value}: F1={f1:.4f}')
            f1_scores.append(f1)

        results[param_name] = (param_values, f1_scores)

    return results


results = run_sweep(
    PARAM_GRID, splits, features, labels, BASE_XGB_PARAMS,
    WEEKS_PER_YEAR, VAL_NO_ONSET, WEIGHT_NO_ONSET, WEIGHT_ONSET, TOLERANCE,
)

results

## Results Plot

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(10, 9))
axs = axs.flatten()
legend_handles, legend_labels = [], []

for i, (param_name, (param_values, f1_scores)) in enumerate(results.items()):
    ax = axs[i]
    ax.scatter(param_values, f1_scores, color='steelblue', marker='o', s=60)

    best_idx   = int(np.argmax(f1_scores))
    best_param = param_values[best_idx]
    best_f1    = f1_scores[best_idx]
    highlight  = ax.scatter(best_param, best_f1, color='crimson', s=120, zorder=5)

    legend_labels.append(f'{param_name}: {best_param}  (F1={best_f1:.3f})')
    legend_handles.append(highlight)

    ax.set_xlabel(param_name, fontsize=12)
    ax.set_ylabel('F1 Score (train)', fontsize=12)
    ax.set_title(param_name, fontsize=13)
    ax.grid(True, linestyle='--', alpha=0.5)

fig.legend(
    handles=legend_handles, labels=legend_labels,
    title='Best Values (red dots)',
    title_fontsize=12, loc='center left',
    bbox_to_anchor=(1.0, 0.5), fontsize=11,
)
plt.suptitle(f'Hyperparameter Sweep — {site_name}', fontsize=16)
plt.tight_layout()

out_dir = OUTPUT_DIR / EXP_NAME / REEF_LOCATION / 'plots'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'{REEF_LOCATION}_hyperparameter_sweep.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {out_path}')